In [0]:
# Service Principal credentials
application_id = "**************************"
authentication_key = "***********************"
tenant_id = "*********************************"

# Set Spark config for ADLS access
spark.conf.set("fs.azure.account.auth.type.petroflowstorage.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.petroflowstorage.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.petroflowstorage.dfs.core.windows.net", application_id)
spark.conf.set("fs.azure.account.oauth2.client.secret.petroflowstorage.dfs.core.windows.net", authentication_key)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.petroflowstorage.dfs.core.windows.net", "https://login.microsoftonline.com/" + tenant_id + "/oauth2/token")

In [0]:
from pyspark.sql.functions import *

gas_bronze = "abfss://bronze@petroflowstorage.dfs.core.windows.net/natural-gas/natural_gas_raw.json"

df_gas_raw = spark.read \
    .option("multiline", "true") \
    .json(gas_bronze)

df_gas_exploded = df_gas_raw \
    .select(explode(col("response.data")).alias("record"))

df_gas_flat = df_gas_exploded.select(
    col("record.period").alias("period"),
    col("record.value").alias("value"),
    col("record.series").alias("series"),
    col("record.series-description").alias("series_description"),
    col("record.units").alias("units"),
    col("record.duoarea").alias("area"),
    col("record.product-name").alias("product_name")
)

df_gas_flat.show(5)
print(f"Raw gas records: {df_gas_flat.count()}")

+-------+-----+--------+--------------------+-----+----+------------+
| period|value|  series|  series_description|units|area|product_name|
+-------+-----+--------+--------------------+-----+----+------------+
|2024-05|12.09|N3010AK3|Alaska Price of N...|$/MCF| SAK| Natural Gas|
|2024-07|14.78|N3010AK3|Alaska Price of N...|$/MCF| SAK| Natural Gas|
|2024-12|11.57|N3010AK3|Alaska Price of N...|$/MCF| SAK| Natural Gas|
|2026-01|12.92|N3010AK3|Alaska Price of N...|$/MCF| SAK| Natural Gas|
|2025-02| 11.6|N3010AK3|Alaska Price of N...|$/MCF| SAK| Natural Gas|
+-------+-----+--------+--------------------+-----+----+------------+
only showing top 5 rows

Raw gas records: 5000


In [0]:
from pyspark.sql.functions import *

df_gas_clean = df_gas_flat \
    .filter(col("value").isNotNull()) \
    .filter(col("period").isNotNull()) \
    .filter(col("value").cast("double") > 0) \
    .withColumn("price_usd", round(col("value").cast("double"), 2)) \
    .withColumn("trade_date", to_date(col("period"), "yyyy-MM")) \
    .withColumn("energy_type", lit("NATURAL_GAS")) \
    .withColumn("source_system", lit("EIA_API")) \
    .withColumn("ingestion_date", current_date()) \
    .select(
        "trade_date",
        "price_usd",
        "series",
        "series_description",
        "units",
        "area",
        "product_name",
        "energy_type",
        "source_system",
        "ingestion_date"
    )

df_gas_clean.show(10)
print(f"Clean gas records: {df_gas_clean.count()}")

+----------+---------+--------+--------------------+-----+----+------------+-----------+-------------+--------------+
|trade_date|price_usd|  series|  series_description|units|area|product_name|energy_type|source_system|ingestion_date|
+----------+---------+--------+--------------------+-----+----+------------+-----------+-------------+--------------+
|2024-05-01|    12.09|N3010AK3|Alaska Price of N...|$/MCF| SAK| Natural Gas|NATURAL_GAS|      EIA_API|    2026-06-04|
|2024-07-01|    14.78|N3010AK3|Alaska Price of N...|$/MCF| SAK| Natural Gas|NATURAL_GAS|      EIA_API|    2026-06-04|
|2024-12-01|    11.57|N3010AK3|Alaska Price of N...|$/MCF| SAK| Natural Gas|NATURAL_GAS|      EIA_API|    2026-06-04|
|2026-01-01|    12.92|N3010AK3|Alaska Price of N...|$/MCF| SAK| Natural Gas|NATURAL_GAS|      EIA_API|    2026-06-04|
|2025-02-01|     11.6|N3010AK3|Alaska Price of N...|$/MCF| SAK| Natural Gas|NATURAL_GAS|      EIA_API|    2026-06-04|
|2024-02-01|    11.08|N3010AK3|Alaska Price of N...|$/MC

In [0]:
gas_silver = "abfss://silver@petroflowstorage.dfs.core.windows.net/natural-gas/"

df_gas_clean.write \
    .mode("overwrite") \
    .parquet(gas_silver)

print("Natural gas written to silver! ✅")

Natural gas written to silver! ✅


In [0]:
df_verify = spark.read.parquet(gas_silver)
df_verify.show(5)
print(f"Total gas records in silver: {df_verify.count()}")

+----------+---------+--------+--------------------+-----+----+------------+-----------+-------------+--------------+
|trade_date|price_usd|  series|  series_description|units|area|product_name|energy_type|source_system|ingestion_date|
+----------+---------+--------+--------------------+-----+----+------------+-----------+-------------+--------------+
|2024-05-01|    12.09|N3010AK3|Alaska Price of N...|$/MCF| SAK| Natural Gas|NATURAL_GAS|      EIA_API|    2026-06-04|
|2024-07-01|    14.78|N3010AK3|Alaska Price of N...|$/MCF| SAK| Natural Gas|NATURAL_GAS|      EIA_API|    2026-06-04|
|2024-12-01|    11.57|N3010AK3|Alaska Price of N...|$/MCF| SAK| Natural Gas|NATURAL_GAS|      EIA_API|    2026-06-04|
|2026-01-01|    12.92|N3010AK3|Alaska Price of N...|$/MCF| SAK| Natural Gas|NATURAL_GAS|      EIA_API|    2026-06-04|
|2025-02-01|     11.6|N3010AK3|Alaska Price of N...|$/MCF| SAK| Natural Gas|NATURAL_GAS|      EIA_API|    2026-06-04|
+----------+---------+--------+--------------------+----